# MMNeedle evaluation quickstart

This notebook shows how to load the [Wang-ML-Lab/MMNeedle](https://huggingface.co/datasets/Wang-ML-Lab/MMNeedle) dataset via the `datasets` library, sample a few examples, and plug them into a simple checker that mimics the repository's `needle.py` evaluation logic.

## Setup
Make sure you have the repo's virtualenv active (or `pip install datasets pillow`).
If you have a private token or want higher rate limits, export `HF_TOKEN` before running the load cell.

In [ ]:
%%capture --no-stderr
import os
import itertools
from datasets import load_dataset
from pprint import pprint


In [ ]:
DATASET_ID = "Wang-ML-Lab/MMNeedle"
SPLIT = "train"
HF_TOKEN = os.environ.get("HF_TOKEN")  # optional
MAX_EXAMPLES = 3


In [ ]:
print("Loading dataset...")
streaming_ds = load_dataset(DATASET_ID, split=SPLIT, streaming=True, token=HF_TOKEN)
examples = list(itertools.islice(streaming_ds, MAX_EXAMPLES))
print(f"Pulled {len(examples)} examples")
examples[0].keys()


In [ ]:
def describe(example):
    summary = {
        'sequence_length': example['sequence_length'],
        'grid': f"{example['grid_rows']}x{example['grid_cols']}",
        'needles_per_query': example['needles_per_query'],
        'has_needle': example['has_needle'],
        'needle_captions': example['needle_captions'],
        'needle_locations': example['needle_locations'],
    }
    pprint(summary)

describe(examples[0])


## Evaluation scaffold
Plug your multimodal model's responses into `evaluate` to compute hit rates.

In [ ]:
from typing import List, Tuple

def normalize_prediction(text: str) -> List[Tuple[int, int, int]]:
    triples = []
    for part in text.split(';'):
        nums = [n.strip() for n in part.split(',')]
        if len(nums) != 3:
            continue
        try:
            triples.append(tuple(int(n) for n in nums))
        except ValueError:
            continue
    return triples

def evaluate(prediction: str, ground_truth: List[dict]) -> float:
    preds = normalize_prediction(prediction)
    gts = [(loc['image_index'] + 1, loc['row'] + 1, loc['col'] + 1) for loc in ground_truth]
    hits = sum(p in gts for p in preds)
    return hits / max(1, len(gts))

# Dummy usage
score = evaluate('1, 1, 1', examples[0]['needle_locations'])
print('dummy score:', score)
